In [1]:
df_clean["participationbegindate_parsed"] = pd.to_datetime(df_clean["participationbegindate"], errors="coerce")

pre_2000 = df_clean[df_clean["participationbegindate_parsed"].dt.year < 2000]
print("Pre-2000 rows:", len(pre_2000))
print()
print("Most common pre-2000 dates:")
print(pre_2000["participationbegindate_parsed"].value_counts().head(10))
print()
print("Pre-2000 rows by state (top 10):")
print(pre_2000["practicestate"].value_counts().head(10))

NameError: name 'pd' is not defined

In [2]:

import pandas as pd
import numpy as np
import os

raw_path = "../data/raw/Medical-Equipment-Suppliers.csv"
df = pd.read_csv(raw_path, dtype=str, low_memory=False)

original_rows = len(df)
original_distinct_providers = df["provider_id"].nunique()
original_true = (df["acceptsassignement"] == "True").sum()
original_acceptance_rate = original_true / original_rows

print("Original rows:", original_rows)
print("Original distinct providers:", original_distinct_providers)
print("Original acceptance rate:", round(original_acceptance_rate * 100, 2), "%")

cleaning_log = []

Original rows: 57197
Original distinct providers: 57197
Original acceptance rate: 51.43 %


In [3]:
columns_to_drop = [
    "businessname", "practicename", "practiceaddress1", "practiceaddress2",
    "practicezip9code", "telephonenumber", "is_contracted_for_cba", "providertypelist"
]

df_clean = df.drop(columns=columns_to_drop)

cleaning_log.append({
    "action": "drop_columns",
    "detail": ", ".join(columns_to_drop),
    "reason": "Excluded by explicit decision: identifying/contact fields, zero-variance field, or 88% missing field",
    "rows_affected": 0
})

print("Remaining columns:", df_clean.columns.tolist())

Remaining columns: ['provider_id', 'acceptsassignement', 'participationbegindate', 'practicecity', 'practicestate', 'specialitieslist', 'supplieslist', 'latitude', 'longitude']


In [4]:
text_cols = ["practicecity", "practicestate", "specialitieslist", "supplieslist"]
for col in text_cols:
    before = df_clean[col].copy()
    df_clean[col] = df_clean[col].str.strip()
    changed = (before != df_clean[col]).sum()
    if changed > 0:
        cleaning_log.append({
            "action": "trim_whitespace",
            "detail": col,
            "reason": "Leading/trailing whitespace standardization",
            "rows_affected": int(changed)
        })

print("Whitespace trimmed.")

Whitespace trimmed.


In [5]:
before = df_clean["practicestate"].copy()
df_clean["practicestate"] = df_clean["practicestate"].str.upper()
changed = (before != df_clean["practicestate"]).sum()
cleaning_log.append({
    "action": "standardize_case",
    "detail": "practicestate uppercased",
    "reason": "Consistent state code formatting",
    "rows_affected": int(changed)
})
print("States standardized. Distinct values:", df_clean["practicestate"].nunique())

States standardized. Distinct values: 55


In [6]:
df_clean["acceptsassignement_bool"] = df_clean["acceptsassignement"].map({"True": True, "False": False})
df_clean["AcceptsNumeric"] = df_clean["acceptsassignement_bool"].astype(int)

unmapped = df_clean["acceptsassignement_bool"].isna().sum()
cleaning_log.append({
    "action": "convert_boolean_target",
    "detail": "acceptsassignement -> acceptsassignement_bool, AcceptsNumeric",
    "reason": "Type conversion for reliable filtering and modeling",
    "rows_affected": int(unmapped)
})
print("Unmapped target values:", unmapped)

Unmapped target values: 0


In [7]:
df_clean["participationbegindate_parsed"] = pd.to_datetime(df_clean["participationbegindate"], errors="coerce")

pre_2000 = df_clean[df_clean["participationbegindate_parsed"].dt.year < 2000]
print("Pre-2000 rows:", len(pre_2000))
print()
print("Most common pre-2000 dates:")
print(pre_2000["participationbegindate_parsed"].value_counts().head(10))
print()
print("Pre-2000 rows by state (top 10):")
print(pre_2000["practicestate"].value_counts().head(10))


Pre-2000 rows: 6915

Most common pre-2000 dates:
participationbegindate_parsed
1998-01-01    4048
1984-10-01     241
1999-12-31     203
1999-01-01     179
1966-07-01     178
1989-10-01     177
1995-01-01     152
1997-01-01     148
1993-01-01      93
1992-01-01      50
Name: count, dtype: int64

Pre-2000 rows by state (top 10):
practicestate
TX    492
NY    407
PA    387
IL    378
OH    345
CA    335
MI    321
WA    237
FL    224
NC    204
Name: count, dtype: int64


In [8]:
# Flag unreliable dates rather than deleting or silently trusting them
placeholder_dates = ["1998-01-01", "1999-01-01", "1997-01-01", "1993-01-01",
                      "1992-01-01", "1999-12-31", "1984-10-01", "1966-07-01", "1989-10-01", "1995-01-01"]

df_clean["date_reliability_flag"] = np.where(
    df_clean["participationbegindate_parsed"].dt.strftime("%Y-%m-%d").isin(placeholder_dates),
    "Suspected placeholder date",
    "Likely genuine date"
)

flag_counts = df_clean["date_reliability_flag"].value_counts()
print(flag_counts)

cleaning_log.append({
    "action": "flag_suspicious_dates",
    "detail": "date_reliability_flag column added",
    "reason": f"6,915 pre-2000 dates found; {4048} rows share the single value 1998-01-01, and several other top values are round year-boundary dates — indicates legacy placeholder dates from system migration, not genuine enrollment dates. Rows preserved, not deleted; flagged for downstream caution.",
    "rows_affected": int(flag_counts.get("Suspected placeholder date", 0))
})

date_reliability_flag
Likely genuine date           51728
Suspected placeholder date     5469
Name: count, dtype: int64


In [9]:
# Look at rows missing specialitieslist - do they have other useful info?
missing_specialty = df_clean[df_clean["specialitieslist"].isna()]
print("Rows missing specialitieslist:", len(missing_specialty))
print()
print("Do these rows have a supplieslist value?")
print(missing_specialty["supplieslist"].notna().sum(), "of", len(missing_specialty), "have supplieslist")
print()
print("Acceptance rate among these rows:", round(missing_specialty["acceptsassignement_bool"].mean() * 100, 2), "%")
print("States represented:", missing_specialty["practicestate"].nunique())
print()

# Same check for missing supplieslist
missing_supply = df_clean[df_clean["supplieslist"].isna()]
print("Rows missing supplieslist:", len(missing_supply))
print("Do these rows have a specialitieslist value?")
print(missing_supply["specialitieslist"].notna().sum(), "of", len(missing_supply), "have specialitieslist")


Rows missing specialitieslist: 752

Do these rows have a supplieslist value?
750 of 752 have supplieslist

Acceptance rate among these rows: 66.22 %
States represented: 50

Rows missing supplieslist: 26
Do these rows have a specialitieslist value?
24 of 26 have specialitieslist


In [10]:
# Fill missing specialty/supply with explicit "Not Specified" - never guess a real category
missing_specialty_count = df_clean["specialitieslist"].isna().sum()
missing_supply_count = df_clean["supplieslist"].isna().sum()

df_clean["specialitieslist"] = df_clean["specialitieslist"].fillna("Not Specified")
df_clean["supplieslist"] = df_clean["supplieslist"].fillna("Not Specified")

cleaning_log.append({
    "action": "fill_missing_categorical",
    "detail": "specialitieslist, supplieslist",
    "reason": "752 rows missing specialitieslist and 26 rows missing supplieslist are otherwise complete, valid provider records (acceptance rate close to baseline, spread across 50 states) - filled with explicit 'Not Specified' rather than dropped or guessed, to preserve real provider records without fabricating a category",
    "rows_affected": int(missing_specialty_count + missing_supply_count)
})

print("Remaining missing specialitieslist:", df_clean["specialitieslist"].isna().sum())
print("Remaining missing supplieslist:", df_clean["supplieslist"].isna().sum())

Remaining missing specialitieslist: 0
Remaining missing supplieslist: 0


In [11]:
# Final reconciliation: compare original raw data to cleaned data
cleaned_rows = len(df_clean)
cleaned_distinct_providers = df_clean["provider_id"].nunique()
cleaned_true = df_clean["acceptsassignement_bool"].sum()
cleaned_acceptance_rate = cleaned_true / cleaned_rows

reconciliation = pd.DataFrame([{
    "original_rows": original_rows,
    "cleaned_rows": cleaned_rows,
    "rows_removed": original_rows - cleaned_rows,
    "rows_modified_estimate": sum(item["rows_affected"] for item in cleaning_log),
    "original_distinct_providers": original_distinct_providers,
    "cleaned_distinct_providers": cleaned_distinct_providers,
    "original_acceptance_rate_pct": round(original_acceptance_rate * 100, 2),
    "cleaned_acceptance_rate_pct": round(cleaned_acceptance_rate * 100, 2),
    "columns_original": len(df.columns),
    "columns_cleaned": len(df_clean.columns),
}])

print(reconciliation.T)

                                     0
original_rows                 57197.00
cleaned_rows                  57197.00
rows_removed                      0.00
rows_modified_estimate         7025.00
original_distinct_providers   57197.00
cleaned_distinct_providers    57197.00
original_acceptance_rate_pct     51.43
cleaned_acceptance_rate_pct      51.43
columns_original                 17.00
columns_cleaned                  13.00


In [12]:
os.makedirs("../data/interim", exist_ok=True)
os.makedirs("../reports/validation", exist_ok=True)

df_clean.to_csv("../data/interim/cms_suppliers_cleaned.csv", index=False)

cleaning_log_df = pd.DataFrame(cleaning_log)
cleaning_log_df.to_csv("../reports/validation/data_cleaning_log.csv", index=False)

reconciliation.to_csv("../reports/validation/data_reconciliation.csv", index=False)

print("Exported:")
print("- data/interim/cms_suppliers_cleaned.csv")
print("- reports/validation/data_cleaning_log.csv")
print("- reports/validation/data_reconciliation.csv")
print()
print("Cleaned dataset shape:", df_clean.shape)
df_clean.head(3)

Exported:
- data/interim/cms_suppliers_cleaned.csv
- reports/validation/data_cleaning_log.csv
- reports/validation/data_reconciliation.csv

Cleaned dataset shape: (57197, 13)


,provider_id,acceptsassignement,participationbegindate,practicecity,practicestate,specialitieslist,supplieslist,latitude,longitude,acceptsassignement_bool,AcceptsNumeric,participationbegindate_parsed,date_reliability_flag
0,34363333,True,2026-07-01,IRVINGTON,NJ,Pharmacy,Epoetin|Immunosuppressive Drugs|Infusion Drugs...,40.735947927803,-74.215432579729,True,1,2026-07-01,Likely genuine date
1,34363327,False,2026-07-01,PHILADELPHIA,PA,Pharmacy,Nebulizer Drugs,39.947301028272,-75.166341976806,False,0,2026-07-01,Likely genuine date
2,34363291,True,2026-06-15,WHARTON,NJ,Pharmacy,Epoetin|Immunosuppressive Drugs|Infusion Drugs...,40.899633588688,-74.582437479208,True,1,2026-06-15,Likely genuine date
